# Module 08: Agentic RAG
This notebook demonstrates how traditional Chroma DB works for RAG pipelines.

## What we'll learn:
- ChromaDB
- OpenAI Embeddings
- RAG using State Machine
- Retrieval, Augment and Generation as steps

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysq    qalite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [5]:
import os
import chromadb
from chromadb.utils import embedding_functions
from chromadb.api.models.Collection import Collection
import pdfplumber
from dotenv import load_dotenv
from typing import TypedDict, List

from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Resource
from lib.llm import LLM
from lib.messages import BaseMessage, UserMessage, SystemMessage

In [6]:
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR)

In [7]:
load_dotenv()

True

In [8]:
sentence_list = [
    "Meta drops multimodal Llama 3.2 — here's why it's such a big deal",
    "Chip giant Nvidia acquires OctoAI, a Seattle startup that helps companies run AI models",
    "Google is bringing Gemini to all older Pixel Buds",
    "The first Intel Battlmage GPU benchmarks have leaked",
    "Dell partners with Nvidia to accelerate AI adoption in telecoms",
]
ids = ["id1", "id2", "id3", "id4", "id5"]

## ChromaDB with Default Embedding Function

In [9]:
chroma_client = chromadb.Client()

In [10]:
collection = chroma_client.create_collection(
    name="demo"
)

In [11]:
collection.add(
    documents=sentence_list,
    ids=ids
)

In [12]:
collection.count()

5

In [13]:
collection.peek(1)

{'ids': ['id1'],
 'embeddings': array([[ 6.06655665e-02, -3.51323076e-02,  6.06436953e-02,
         -5.11926487e-02,  1.13580227e-01, -1.88892838e-02,
         -2.68527884e-02,  5.48634380e-02,  3.23644355e-02,
          5.42442799e-02, -4.04198654e-02, -1.90558434e-02,
         -5.97919747e-02,  2.56032161e-02,  8.48460123e-02,
          4.12196517e-02,  3.95206064e-02, -4.00091223e-02,
         -7.66606480e-02,  2.78291572e-02,  5.38355410e-02,
         -1.35247586e-02,  9.65649858e-02, -3.04362196e-02,
          6.61454443e-03,  7.21731409e-02, -9.53866541e-02,
         -2.75959242e-02,  7.86795467e-03, -6.68519363e-02,
         -1.27341459e-02,  1.21337987e-01, -6.66138232e-02,
         -3.28670666e-02, -6.49284273e-02, -1.61902420e-02,
         -3.32962652e-03,  8.04080814e-02, -3.84503976e-02,
          1.37208350e-04,  3.72594525e-03,  4.83830906e-02,
         -3.68936890e-06, -4.51370440e-02, -1.37449000e-02,
         -7.15254173e-02,  1.01806214e-02, -4.23030593e-02,
         

In [14]:
collection.query(
    query_texts=["gadget"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)

{'ids': [['id3', 'id1']],
 'embeddings': None,
 'documents': [['Google is bringing Gemini to all older Pixel Buds',
   "Meta drops multimodal Llama 3.2 — here's why it's such a big deal"]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[1.5251758098602295, 1.7548508644104004]]}

In [17]:
result = collection.query(
    query_texts=["gadget"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)

result['documents'][0]

['Google is bringing Gemini to all older Pixel Buds',
 "Meta drops multimodal Llama 3.2 — here's why it's such a big deal"]

In [18]:
print(collection._embedding_function.name())

default


In [19]:
size = len(collection.peek(1)['embeddings'][0])
print(f"Size of the embeddings array: {size}")


Size of the embeddings array: 384


## OpenAI Embeddings

In [20]:
chroma_client.delete_collection(name="demo")

In [24]:
embeddings_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [25]:
collection = chroma_client.create_collection(
    name="demo",
    embedding_function=embeddings_fn
)

InternalError: Collection [demo] already exists

In [26]:
collection.add(
    documents=sentence_list,
    ids=ids
)

In [27]:
collection.query(
    query_texts=["gadget"],
    n_results=2,
    include=['metadatas', 'documents', 'distances']
)

{'ids': [['id3', 'id4']],
 'embeddings': None,
 'documents': [['Google is bringing Gemini to all older Pixel Buds',
   'The first Intel Battlmage GPU benchmarks have leaked']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.2330048680305481, 0.2433934211730957]]}

In [28]:
print(collection._embedding_function.name())

openai


In [29]:
size = len(collection.peek(1)['embeddings'][0])
print(f"Size of the embeddings array: {size}")

Size of the embeddings array: 1536


## RAG

**Load**

In [30]:
file_path = "GlobalEVOutlook2025.pdf"
documents = []
page_nums = []

In [31]:
with pdfplumber.open(file_path) as pdf:
    for num, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()
        if text:
            documents.append(text)
            page_nums.append(str(num))


In [32]:
collection = chroma_client.create_collection(
    name="traditional_rag",
    embedding_function=embeddings_fn
)

In [33]:
collection.add(
    documents=documents,
    ids=page_nums
)

**State Machine**

In [34]:
class State(TypedDict):
    messages: List[BaseMessage]
    question: str
    documents: List[str]
    answer: str

**RAG: Retrieve**

In [36]:
def retrieve(state:State, resource:Resource):
    question = state["question"]
    collection:Collection = resource.vars.get("collection")
    results = collection.query(
        query_texts=[question],
        n_results=3,
        include=['documents']
    )
    retrieved_docs = results['documents'][0]
    
    return {"documents": retrieved_docs}

**RAG: Augment**

In [37]:
def augment(state:State):
    question = state["question"]
    documents = state["documents"]
    context = "\n\n".join(documents)

    messages = [
        SystemMessage(content="You are an assistant for question-answering tasks."),
        UserMessage(
            content=(
                "Use the following pieces of retrieved context to answer the question. "
                "If you don't know the answer, just say that you don't know. "
                f"\n# Question: \n-> {question} "
                f"\n# Context: \n-> {context} "
                "\n# Answer: "
            )
        )
    ]

    return {"messages": messages}

**RAG: Generate**

In [38]:
def generate(state:State, resource:Resource):
    llm:LLM = resource.vars.get("llm")
    ai_message = llm.invoke(state["messages"])
    return {
        "answer": ai_message.content, 
        "messages": state["messages"] + [ai_message],
    }

In [39]:
workflow = StateMachine(State)

In [40]:
# Create steps
entry = EntryPoint()
retrieve_step = Step("retrieve", retrieve)
augment_step = Step("augment", augment)
generate_step = Step("generate", generate)
termination = Termination()
        
workflow.add_steps(
    [
        entry, 
        retrieve_step, 
        augment_step, 
        generate_step, 
        termination
    ]
)

In [41]:
# Add transitions
workflow.connect(entry, retrieve_step)
workflow.connect(retrieve_step, augment_step)
workflow.connect(augment_step, generate_step)
workflow.connect(generate_step, termination)

In [42]:
llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

In [43]:
resource = Resource(
    vars = {
        "llm": llm,
        "collection": collection,
    }
)

In [44]:
initial_state: State = {
    "question": "What was the number of electric car sales and their market share in Brazil in 2024?",
}

In [45]:
run_object = workflow.run(initial_state, resource)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__


In [46]:
run_object.get_final_state()["answer"]

'In 2024, Brazil had nearly 125,000 electric car sales, which represented a market share of 6.5%.'